In [21]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

In [22]:
X_raw = np.load('X_raw.npy')
Y     = np.load('Y.npy')

# Separamos la señal ECG de los datos demográficos
X_signal = X_raw[:, :-2].reshape(X_raw.shape[0], 1, -1)  # (N, 1, 3300)
X_demo   = X_raw[:, -2:]                                   # (N, 2) → edad y sexo

print(f"Shape señal: {X_signal.shape}")
print(f"Shape demo:  {X_demo.shape}")
print(f"Shape Y:     {Y.shape}")

MemoryError: Unable to allocate 389. MiB for an array with shape (50985968,) and data type float64

In [ ]:
# Dividimos en train y test manteniendo las dos entradas alineadas
indices = np.arange(len(X_signal))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=42)

X_signal_train, X_signal_test = X_signal[idx_train], X_signal[idx_test]
X_demo_train,   X_demo_test   = X_demo[idx_train],   X_demo[idx_test]
Y_train,        Y_test        = Y[idx_train],         Y[idx_test]

# Escalamos la señal — reshape a 2D para el scaler y luego volvemos a 3D
N_train, C, L = X_signal_train.shape
N_test        = X_signal_test.shape[0]

scaler_signal = StandardScaler()
X_signal_train = scaler_signal.fit_transform(
    X_signal_train.reshape(N_train, -1)).reshape(N_train, C, L)
X_signal_test  = scaler_signal.transform(
    X_signal_test.reshape(N_test, -1)).reshape(N_test, C, L)

# Escalamos también edad y sexo
scaler_demo  = StandardScaler()
X_demo_train = scaler_demo.fit_transform(X_demo_train)
X_demo_test  = scaler_demo.transform(X_demo_test)

print(f"Train señal: {X_signal_train.shape} | Train demo: {X_demo_train.shape}")
print(f"Test  señal: {X_signal_test.shape}  | Test  demo: {X_demo_test.shape}")

Train señal: (13587, 1, 3000) | Train demo: (13587, 2)
Test  señal: (3397, 1, 3000)  | Test  demo: (3397, 2)


In [ ]:
# Convertimos todo a tensores de PyTorch
X_signal_train_t = torch.tensor(X_signal_train, dtype=torch.float32)
X_signal_test_t  = torch.tensor(X_signal_test,  dtype=torch.float32)
X_demo_train_t   = torch.tensor(X_demo_train,   dtype=torch.float32)
X_demo_test_t    = torch.tensor(X_demo_test,    dtype=torch.float32)
Y_train_t        = torch.tensor(Y_train,        dtype=torch.float32)

# Cada muestra tiene tres componentes: señal, demo e etiqueta
train_loader = DataLoader(
    TensorDataset(X_signal_train_t, X_demo_train_t, Y_train_t),
    batch_size=32, shuffle=True
)
test_loader = DataLoader(
    TensorDataset(X_signal_test_t, X_demo_test_t,
                  torch.zeros(N_test)),
    batch_size=32, shuffle=False
)

# Pesos para balanceo de clases
total      = Y_train.shape[0]
pos_counts = Y_train.sum(axis=0)
neg_counts = total - pos_counts
pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-8), dtype=torch.float32)

In [ ]:
class CNN1D_Demo(nn.Module):
    def __init__(self, n_canales, longitud_señal, n_demo, n_clases):
        super().__init__()

        self.rama_cnn = nn.Sequential(
            nn.Conv1d(n_canales, 32, kernel_size=25, padding=12),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.2),

            nn.Conv1d(32, 64, kernel_size=15, padding=7),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.2),

            nn.Conv1d(64, 128, kernel_size=9, padding=4),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.2),
        )

        # Calculamos el tamaño de salida de la CNN automáticamente
        # pasando un tensor de prueba por la rama CNN
        with torch.no_grad():
            prueba = torch.zeros(1, n_canales, longitud_señal)
            salida_cnn = self.rama_cnn(prueba)
            tamaño_cnn = salida_cnn.view(1, -1).shape[1]

        print(f"Tamaño salida CNN: {tamaño_cnn}")

        self.flatten = nn.Flatten()

        self.fc_cnn = nn.Sequential(
            nn.Linear(tamaño_cnn, 128),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.rama_demo = nn.Sequential(
            nn.Linear(n_demo, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU()
        )

        self.clasificador = nn.Sequential(
            nn.Linear(128 + 16, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, n_clases)
        )

    def forward(self, x_signal, x_demo):
        cnn_out   = self.fc_cnn(self.flatten(self.rama_cnn(x_signal)))
        demo_out  = self.rama_demo(x_demo)
        fusionado = torch.cat([cnn_out, demo_out], dim=1)
        return self.clasificador(fusionado)

# Ahora pasamos la longitud real de la señal al constructor
longitud_señal = X_signal_train.shape[2]
modelo = CNN1D_Demo(n_canales=1, longitud_señal=longitud_señal, n_demo=2, n_clases=Y_train.shape[1])
#print(modelo)

Tamaño salida CNN: 48000


In [ ]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(modelo.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

EPOCHS    = 100
PACIENCIA = 15
mejor_perdida     = float('inf')
epochs_sin_mejora = 0
mejor_estado      = None

print("=== CNN 1D + Demografía ===")
for epoch in range(EPOCHS):
    modelo.train()
    perdida_total = 0
    for X_sig, X_dem, Y_batch in train_loader:
        optimizer.zero_grad()
        output = modelo(X_sig, X_dem)  # pasamos las dos entradas
        loss   = criterion(output, Y_batch)
        loss.backward()
        optimizer.step()
        perdida_total += loss.item()

    perdida_media = perdida_total / len(train_loader)
    scheduler.step(perdida_media)

    if (epoch + 1) % 10 == 0:
        lr_actual = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1}/{EPOCHS} - Pérdida: {perdida_media:.4f} - lr: {lr_actual:.6f}")

    if perdida_media < mejor_perdida - 0.001:
        mejor_perdida     = perdida_media
        epochs_sin_mejora = 0
        mejor_estado      = {k: v.clone() for k, v in modelo.state_dict().items()}
    else:
        epochs_sin_mejora += 1
        if epochs_sin_mejora >= PACIENCIA:
            print(f"Early stopping en epoch {epoch+1}")
            break

modelo.load_state_dict(mejor_estado)
print("Entrenamiento completado")

=== CNN 1D + Demografía ===


KeyboardInterrupt: 

In [ ]:
modelo.eval()
all_preds = []
with torch.no_grad():
    for X_sig, X_dem, _ in test_loader:
        output = torch.sigmoid(modelo(X_sig, X_dem))
        all_preds.append((output >= 0.5).float())

print("=== Resultados CNN 1D + Demografía ===")
print(classification_report(Y_test, torch.cat(all_preds).numpy(), zero_division=0))

=== Resultados CNN 1D + Demografía ===
              precision    recall  f1-score   support

           0       0.30      0.79      0.43       834
           1       0.63      0.57      0.60      1421
           2       0.34      0.79      0.48       892
           3       0.16      0.77      0.26       416
           4       0.29      0.67      0.41       757

   micro avg       0.32      0.70      0.44      4320
   macro avg       0.34      0.72      0.44      4320
weighted avg       0.40      0.70      0.48      4320
 samples avg       0.40      0.66      0.47      4320

